<a id="inizio"></a>
# 11 · Capstone: il carico del Nord e la temperatura

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Avanzata &nbsp;·&nbsp; ⏱ ~110 min**

Due anni di carico elettrico della zona Nord e la temperatura di Milano: come il carico dipende dall'ora, dal giorno della settimana e dalla temperatura.

**In questo notebook impariamo a**

- caricare, unire e pulire una serie temporale reale, con i salti dell'ora legale
- costruire la serie giornaliera e i profili medi per ora e per giorno della settimana
- unire il carico con la temperatura di Open-Meteo e leggere la relazione tra i due

Prima di questo: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb) &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/load_total_north_hourly_2025.xlsx`, `../Dati/fallback/meteo_milano_2024_2025.json`

</div>

<a id="indice"></a>
**Indice**

- [1. Caricamento dei dati e unione](#sez-1)
    - [Step 1 · Caricamento dei dati e unione](#es-1)
- [2. Colonna temporale e prima visualizzazione](#sez-2)
    - [Step 2 · Preparazione della colonna temporale e prima visualizzazione](#es-2)
- [3. Ora legale: duplicati e buchi](#sez-3)
    - [Step 3 · Pulizia dei duplicati dell'ora legale](#es-3)
- [4. Aggregazioni temporali e profili medi](#sez-4)
    - [Step 4 · Aggregazioni temporali e profili medi](#es-4)
- [5. Recupero e integrazione della temperatura](#sez-5)
    - [Step 5 · Recupero e integrazione della temperatura](#es-5)
- [6. Carico e temperatura](#sez-6)
    - [Step 6 · Carico e temperatura](#es-6)

I dati vengono dal [Download Center di Terna](https://dati.terna.it/en/download-center): il **carico
elettrico della zona Nord** negli anni 2024 e 2025. Nei dataset Terna il carico è la potenza richiesta
al sistema elettrico in ciascun intervallo temporale, una grandezza centrale per la pianificazione e la
gestione della rete di trasmissione.

Accanto al carico integriamo la **temperatura** di Milano, presa tramite API da
[Open-Meteo](https://open-meteo.com). L'obiettivo è descrivere come il carico del Nord dipende dall'ora
del giorno, dal giorno della settimana e dalla temperatura.

Per i titoli e le etichette dei grafici si può usare Copilot; il codice degli step lo scriviamo noi.

<a id="sez-1"></a>
## 1. Caricamento dei dati e unione

<a id="es-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 1 · Caricamento dei dati e unione**

In questo step costruiamo una singola serie storica continua a partire dai dati grezzi.

**Cosa fare.**

1. Leggi i due file Excel presenti nella cartella `../Dati/` in `df_2024` e `df_2025`:
   `load_total_north_hourly_2024.xlsx` e `load_total_north_hourly_2025.xlsx`.
2. Unisci i due dataset **per righe** in `df`, mantenendo lo stesso schema di colonne, in modo da
   ottenere una serie unica che copra l'intero periodo 2024-2025.
3. Controlla dimensione del dataset, prima e ultima data, quante righe ci sono in media per giorno
   (in `righe_giorno`) e i valori mancanti per colonna. Il file si chiama `hourly`: la frequenza è
   oraria?

Risultato atteso: 70.176 righe e 4 colonne, nessun valore mancante.

</div>

In [ ]:
import pandas as pd

df = ...

righe_giorno = ...
df.isna().sum()

In [ ]:
# Verifica: esegui senza modificare
assert df.shape == (70176, 4), "❌ df: 70.176 righe e 4 colonne, i due anni uno sotto l'altro"
assert df.index[-1] == 70175, "❌ df: l'indice va rifatto da 0, con ignore_index=True in pd.concat"
assert round(righe_giorno) == 96, "❌ righe_giorno: righe totali diviso il numero di giorni diversi"
print("✅ Step 1 completato")

96 righe al giorno: i dati sono quartorari, un valore ogni 15 minuti, anche se il nome del file dice
`hourly`.

<a id="sez-2"></a>
## 2. Colonna temporale e prima visualizzazione

<a id="es-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 2 · Preparazione della colonna temporale e prima visualizzazione**

In questo step rendiamo la colonna data utilizzabile come variabile temporale e facciamo una prima ispezione visiva della serie.

**Cosa fare.**

1. Controlla con `df.dtypes` che `Date` sia già `datetime`; se fosse testo, `pd.to_datetime` la
   converte.
2. Ordina il dataset per data (i file partono dall'ultimo quarto d'ora dell'anno) e reimposta
   l'indice per avere un ordinamento pulito.
3. Importa Plotly e visualizza **entrambe** le serie nel tempo: `Total Load [MW]` e
   `Forecast Total Load [MW]`.
4. Dopo aver verificato che le due curve siano coerenti, elimina le colonne che non useremo:
   `Forecast Total Load [MW]` e `Bidding Zone`.

Al termine di questo step deve rimanere una serie storica con una colonna temporale in formato
`datetime` e una sola colonna di valori, `Total Load [MW]`.

</div>

In [ ]:
import plotly.express as px

df["Date"] = ...
df = ...

fig = ...
fig.show()

df = ...
df.head()

In [ ]:
# Verifica: esegui senza modificare
assert list(df.columns) == ["Date", "Total Load [MW]"], "❌ df: devono restare solo Date e Total Load [MW]"
assert df["Date"].is_monotonic_increasing, "❌ df: ordina per Date"
assert df.loc[0, "Date"] == pd.Timestamp("2024-01-01 00:00"), "❌ df: la riga 0 deve essere il primo quarto d'ora del 2024; dopo sort_values serve reset_index(drop=True)"
print("✅ Step 2 completato")

<a id="sez-3"></a>
## 3. Ora legale: duplicati e buchi

Prima di aggregare controlliamo il passo tra un timestamp e il successivo. Due giorni all'anno qualcosa
non torna: l'ultima domenica di ottobre e l'ultima domenica di marzo, quando cambia l'ora.

Proviamo prima a dare alle date il fuso orario italiano e a lasciare a pandas il riconoscimento
dell'ora ripetuta. La cella dà errore apposta: leggiamo l'ultima riga.

In [ ]:
df["Date"].dt.tz_localize("Europe/Rome", ambiguous="infer")

`There are 4 dst switches when there should only be 1`: le due copie di ogni quarto d'ora ripetuto
sono intrecciate, e pandas vede quattro cambi d'ora invece di uno. Per questa analisi basta l'ora
locale senza fuso: togliamo i doppioni con `drop_duplicates`.

<a id="es-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 3 · Pulizia dei duplicati dell'ora legale**

In questo step lasciamo una riga per timestamp e contiamo i quarti d'ora che mancano.

**Cosa fare.**

1. In `passi` metti la differenza tra ogni timestamp e il precedente (`diff()`) e conta quante volte
   compare ogni valore con `value_counts()`. Qual è il passo normale? Cosa sono gli altri due valori?
2. In `duplicati` metti le righe con un timestamp ripetuto, tutte le copie
   (`duplicated(keep=False)`). Che giorni sono? Che ore?
3. Ordina per `Date`, togli i duplicati con `drop_duplicates(subset="Date")` e reimposta l'indice.
4. In `buchi` metti le righe che arrivano dopo un salto più lungo di 15 minuti. Che giorni sono?

</div>

In [ ]:
passi = ...

duplicati = ...

df = ...

buchi = ...
buchi

In [ ]:
# Verifica: esegui senza modificare
assert len(duplicati) == 16, "❌ duplicati: 16 righe, 8 timestamp ripetuti in due copie (keep=False le tiene tutte)"
assert len(df) == 70168 and df["Date"].is_unique, "❌ df: 70.168 righe, una per timestamp"
assert len(buchi) == 2, "❌ buchi: 2 righe, le 3:00 delle due ultime domeniche di marzo"
print("✅ Step 3 completato")

<a id="sez-4"></a>
## 4. Aggregazioni temporali e profili medi

Useremo `Date` come colonna temporale e `Total Load [MW]` come variabile di interesse.

<a id="es-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 4 · Aggregazioni temporali e profili medi**

In questo step analizziamo la serie quartoraria cambiando punto di vista sul tempo. Costruiamo una
serie giornaliera e osserviamo i pattern medi legati all'ora del giorno e al giorno della settimana.

**Cosa fare.**

**Serie giornaliera (somma)**

1. Parti dal DataFrame quartorario `df`.
2. Imposta la colonna `Date` come indice temporale in `df_temp`.
3. Aggrega per giorno usando `resample("D")` e calcola la **somma** del carico.
4. Riporta `Date` come colonna e salva il risultato in `df_daily`. Aggiungi la colonna
   `Energia [MWh]`: la somma divisa per 4, perché un quarto d'ora a P MW vale P/4 MWh.
5. Visualizza la serie giornaliera con Plotly.

**Profilo medio giornaliero (media per ora del giorno)**

6. Torna alla serie quartoraria `df`: estrai l'**ora del giorno** dalla colonna `Date` e salvala in
   una nuova colonna `hour`.
7. Raggruppa per `hour` e calcola la **media** del carico in `profilo_orario`.
8. Visualizza il profilo medio giornaliero con Plotly.

**Profilo medio settimanale (media per giorno della settimana)**

9. Sempre partendo da `df`, estrai il **giorno della settimana** dalla colonna `Date` e salvalo in
   una nuova colonna `weekday`, con la convenzione `0 = lunedì, …, 6 = domenica`.
10. Raggruppa per `weekday` e calcola la **media** del carico in `profilo_settimanale`.
11. Visualizza il profilo medio settimanale con Plotly.

Guarda i tre grafici: in che periodi l'energia giornaliera scende? A che ore il carico è più alto,
e di quanto cala nel weekend?

</div>

In [ ]:
# serie giornaliera
df_daily = ...

# profilo medio giornaliero
profilo_orario = ...

# profilo medio settimanale
profilo_settimanale = ...

In [ ]:
# Verifica: esegui senza modificare
assert len(df_daily) == 731 and df_daily["Energia [MWh]"].between(250_000, 700_000).all(), "❌ df_daily: 731 giorni, con l'energia tra 250.000 e 700.000 MWh (somma dei quartorari divisa per 4)"
assert profilo_orario.shape == (24, 2), "❌ profilo_orario: 24 righe (le ore) e 2 colonne, hour e Total Load [MW]"
assert len(profilo_settimanale) == 7 and profilo_settimanale["Total Load [MW]"].idxmin() == 6, "❌ profilo_settimanale: 7 righe, con la domenica (6) come giorno più leggero"
print("✅ Step 4 completato")

<a id="sez-5"></a>
## 5. Recupero e integrazione della temperatura

Aggiungiamo la **temperatura a 2 metri** ricostruita per Milano. È una semplificazione: usiamo Milano
come punto "medio" per il Nord Italia. L'API
[Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) di Open-Meteo non chiede
alcuna chiave e restituisce i dati orari su un intervallo di date.

<a id="es-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 5 · Recupero e integrazione della temperatura**

Ora dobbiamo integrare il dato di temperatura con il dato di carico. Per prima cosa va risolta la
differente granularità: la temperatura è oraria, il carico quartorario (da 1 ora a 15 minuti).

**Cosa fare.**

1. Chiedi all'archivio storico di Open-Meteo la temperatura oraria di Milano (latitudine 45.4642,
   longitudine 9.19) dal primo all'ultimo giorno del carico. URL
   `https://archive-api.open-meteo.com/v1/archive`; parametri `latitude`, `longitude`, `start_date`
   ed `end_date` come testo `AAAA-MM-GG`, `hourly="temperature_2m"`, `timezone="Europe/Rome"`.
   Salva in `orario` la chiave `hourly` della risposta.
2. Nella seconda cella trasforma `orario` in `meteo` con `pd.DataFrame`.
3. Converti `time` in datetime, rinomina le colonne in `Date` e `Temperature_C`, imposta `Date` come
   indice e ordinalo con `sort_index()`.
4. Fai il resample a quartorario per poter integrare con il carico (`resample("15min")` +
   `interpolate(method="linear")`), riporta `Date` come colonna e salva il risultato in `meteo_15`.
5. Fai il merge con il DataFrame del carico `df`, tenendo tutte le sue righe, e salva il risultato in
   `df_full`.
6. Mancano i dati dell'ultima ora del 2025: riempi la temperatura con `ffill()`.
7. Visualizza la temperatura con `px.line()`.

</div>

In [ ]:
import requests

start_date = ...
end_date = ...
url = "https://archive-api.open-meteo.com/v1/archive"
params = ...

orario = ...

Se la rete non risponde, la cella qui sotto carica da `../Dati/fallback/` temperature di esempio
(non misurate); se l'API ha risposto, saltala.

In [ ]:
import json

with open("../Dati/fallback/meteo_milano_2024_2025.json", encoding="utf-8") as f:
    orario = json.load(f)["hourly"]
len(orario["time"])

In [ ]:
# punti 2-7: da orario a df_full
meteo = ...

meteo_15 = ...

df_full = ...

In [ ]:
# Verifica: esegui senza modificare
assert list(meteo_15.columns) == ["Date", "Temperature_C"], "❌ meteo_15: due colonne, Date e Temperature_C (dopo resample serve reset_index)"
assert len(df_full) == len(df), "❌ df_full: il merge con how=\"left\" tiene tutte le righe di df"
assert df_full["Temperature_C"].notna().all(), "❌ df_full: restano temperature mancanti, usa ffill()"
print("✅ Step 5 completato")

<a id="sez-6"></a>
## 6. Carico e temperatura

<a id="es-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 6 · Carico e temperatura**

Un punto per giorno: temperatura media sulle x, carico medio sulle y, un colore per mese. Dalla forma della nuvola di punti si legge come il carico dipende dalla temperatura.

**Cosa fare.**

1. Porta `df_full` a medie giornaliere in `giornaliero`: `Date` come indice, le colonne
   `Total Load [MW]` e `Temperature_C`, `resample("D").mean()`, poi `reset_index()`.
2. Aggiungi la colonna `mese` con il nome del mese (`dt.month_name()`): così lo scatter ha un colore
   per mese invece di una scala continua.
3. Disegna lo scatter con `px.scatter`: temperatura sulle x, carico sulle y, `color="mese"`. Che
   forma ha? A che temperatura il carico è più basso?
4. Completa le cinque frasi della cella dopo la verifica, con i numeri e i grafici degli step.

</div>

In [ ]:
giornaliero = ...

fig = ...
fig.show()

In [ ]:
# Verifica: esegui senza modificare
assert len(giornaliero) == 731, "❌ giornaliero: una riga per giorno, 731"
assert {"Total Load [MW]", "Temperature_C", "mese"} <= set(giornaliero.columns), "❌ giornaliero: servono le colonne Total Load [MW], Temperature_C e mese"
assert giornaliero["mese"].nunique() == 12, "❌ mese: i dodici mesi, da dt.month_name()"
print("✅ Step 6 completato")

**Il carico del Nord, 2024-2025**

1. I dati sono ...: ... righe pulite, ... doppioni di ottobre tolti, ... quarti d'ora di marzo mancanti.
2. L'energia giornaliera va da circa ... a circa ... MWh; i giorni più leggeri sono ...
3. Nella giornata media il carico è minimo alle ... e massimo alle ...
4. La domenica il carico medio sta intorno a ... MW, un giorno feriale intorno a ... MW.
5. Il carico è più basso intorno a ... °C e sale con il freddo e con il caldo; pesa di più ...

Fonte della temperatura: ... (Open-Meteo oppure i dati di esempio di `../Dati/fallback/`).

---

**Fine del notebook 11.**